In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY=os.getenv("LANGSMITH_API_KEY")
LANGCHAIN_PROJECT=os.getenv("LANGCHAIN_PROJECT")
TAVILI_API_KEY=os.getenv("TAVILI_API_KEY")

In [92]:
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["GROQ_API_KEY"]= GROQ_API_KEY
os.environ["LANGSMITH_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"]=LANGCHAIN_PROJECT

In [7]:
from langchain_text_splitters  import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma

c:\Users\Admin\Documents\ML_Projects\Agentic_AI\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="model/embeddin-001")

from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-1.0-pro")


In [13]:
from langchain_groq import ChatGroq
import os
llm = ChatGroq(model_name="llama-3.1-8b-instant")

In [14]:
while True:
    question=input("type your question. if you want to quit the chat write quit")
    if question !="quit":
        print(llm.invoke(question).content)
    else:
        print("goodbye take care yourself")
        break

**Machine Learning (ML)** is a subset of **Artificial Intelligence (AI)** that involves the use of algorithms and statistical models to enable computers to learn from data, make decisions, and improve their performance over time.

**Key Characteristics of Machine Learning:**

1. **Data-driven**: ML algorithms rely on large amounts of data to learn and improve.
2. **Automated**: The process of learning and improving is automated, with the computer making decisions based on the data.
3. **Adaptive**: ML algorithms can adjust to new data and changing conditions.
4. **Improved performance**: The system becomes more accurate and efficient over time.

**Types of Machine Learning:**

1. **Supervised Learning**: The algorithm is trained on labeled data, where the correct output is already known.
2. **Unsupervised Learning**: The algorithm is trained on unlabeled data, where the goal is to identify patterns or structure.
3. **Reinforcement Learning**: The algorithm learns through trial and erro

In [16]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import AIMessage

In [17]:
store = {}

In [18]:

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [19]:

config = {"configurable": {"session_id": "firstchat"}}

In [20]:
model_with_memory = RunnableWithMessageHistory(llm, get_session_history)

In [21]:
model_with_memory.invoke(("Hi! I'm Anand"),config=config).content

'Nice to meet you, Anand. How can I assist you today?'

In [23]:
model_with_memory.invoke(("tell me what is my name?"),config=config).content

'Your name is Anand.'

In [31]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough , RunnableLambda
from langchain_core.output_parsers import StrOutputParser

### Reading the txt files from source directory

loader = DirectoryLoader('../data', glob="./*.txt", loader_cls=TextLoader)
docs = loader.load()

### Creating Chunks using RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    length_function=len
)
new_docs = text_splitter.split_documents(documents=docs)
doc_strings = [doc.page_content for doc in new_docs]

# BGE embeddings
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

model_name = "BAAI/bge-base-en-v1.5"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True}

embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

### Creating Retriever using Vector DB
db = Chroma.from_documents(new_docs, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 4})

C:\Users\Admin\AppData\Local\Temp\ipykernel_5760\4088430924.py:30: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 747.78it/s]


In [32]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)

In [33]:
retrieval_chain = (
    RunnableParallel({"context": retriever, "question": RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
)


In [34]:
question ="what is llama3? can you highlight 3 important points?"
print(retrieval_chain.invoke(question))

Based on the given context, here's what I can infer about Llama 3:

Llama 3 appears to be a version of a large language model developed by Meta. From the context, I can highlight the following three important points:

1. **Release**: Llama 3 seems to have been released in April 2024, as mentioned in the first document.
2. **Model versions**: There is a mention of an "8B parameter version" of Llama 3, which suggests that Llama 3 comes in different model sizes or configurations, with varying numbers of parameters.
3. **Integration with services**: Llama 3 is being used by Meta's services, such as a website and another service (though the name of the second service is not specified). This indicates that Llama 3 is being applied in various use cases to improve user experience.


### Let's Start with Tools and Agents

In [36]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [38]:
api_wrapper  = WikipediaAPIWrapper(top_k_results=5, doc_content_chars_max=100)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [39]:
tool.name

'wikipedia'

In [40]:
tool.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [41]:
tool.args

{'query': {'description': 'query to look up on wikipedia',
  'title': 'Query',
  'type': 'string'}}

In [42]:
tool.return_direct

False

In [43]:
print(tool.run({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of 


In [44]:
tool.run("langchain")

'Page: LangChain\nSummary: LangChain is a software framework that helps facilitate the integration of '

In [47]:
from pydantic import BaseModel, Field

In [48]:
class WikiInputs(BaseModel):
    query : str = Field(description="query to look up in Wikipedia, should be 3 or less words")

In [50]:
from langchain_core.tools import Tool
from pydantic import BaseModel, Field
from langchain_community.utilities import WikipediaAPIWrapper

class WikiInputs(BaseModel):
    query: str = Field(description="search query for wikipedia")

wiki = WikipediaAPIWrapper()

tool = Tool(
    name="wiki-tool",
    description="look up things in wikipedia",
    func=wiki.run,
    args_schema=WikiInputs
)

In [51]:
tool.name

'wiki-tool'

In [52]:
tool.description

'look up things in wikipedia'

In [53]:
tool.args

{'query': {'description': 'search query for wikipedia',
  'title': 'Query',
  'type': 'string'}}

In [54]:
tool.return_direct

False

In [55]:
print(tool.run("langchain"))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Vector database
Summary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vector space. Vector databases typically implement approximate nearest neighbor algorithms so users can search for records semantically similar to a given input, unlike traditional databases which primarily look up records by exact match. Use-cases for vector databases include similarity search, semantic search, multi-modal search, recommendations engines, object detection, and retrieval-augmented generation (RAG).
Vector embeddings are mathematical representations of data in a high-dimensional s

### youtube search tool

In [ ]:
from langchain_community.tools import YouTubeSearchTool

In [68]:
tool = YouTubeSearchTool()
print(tool.name)
print(tool.description)
tool.run("MotorOctane")

youtube_search
search for youtube videos associated with a person. the input to this tool should be a comma separated list, the first part contains a person name and the second a number that is the maximum number of video results to return aka num_results. the second part is optional


"['https://www.youtube.com/watch?v=TJhGYqDWaT8&pp=ygULTW90b3JPY3RhbmU%3D', 'https://www.youtube.com/watch?v=L1u1xbwUvts&pp=ygULTW90b3JPY3RhbmU%3D']"

In [69]:
from langchain_community.tools.tavily_search import TavilySearchResults

c:\Users\Admin\Documents\ML_Projects\Agentic_AI\venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [93]:
tool = TavilySearchResults()
tool.invoke({"query": "Who won 2026 T20 cricket worldcup"})

[{'title': "2026 Men's T20 World Cup - Wikipedia",
  'url': 'https://en.wikipedia.org/wiki/2026_Men%27s_T20_World_Cup',
  'content': '(4 overs) India A won by 38 runs DY Patil Stadium, Navi Mumbai Umpires: Rohan Pandit (Ind) and Vinod Seshan (Ind) Player of the match: Narayan Jagadeesan (Ind)  United States won the toss and elected to field.    Warm-up match 3 2 February 2026(2026-02-02) 19:00+05:30 (N) Scorecard ItalyImage 88 156/4 (20 overs)vImage 89Canada 146/6 (20 overs) JJ Smuts 49 (37) Dilpreet Bajwa 1/24 (3 overs)Dilpreet Bajwa 37 (34) Grant Stewart "Grant Stewart (cricketer)") 1/18 (3 overs) Italy won by 10 runs M. A. Chidambaram Stadium, Chennai Umpires: Saidharshan Kumar (Ind) and Madanagopal Kuppuraj (Ind) Player of the match: JJ Smuts (Ita)  Canada won the toss and elected to field.    Warm-up match 4 3 February 2026(2026-02-03) 13:00+05:30 Scorecard Sri Lanka AImage 90: flag 145/9 (20 overs)vImage 91Oman [...] Sri Lanka AImage 90: flag 145/9 (20 overs)vImage 91Oman 146/5 (

In [98]:
tool.invoke({"query": "Highest selling suv in India in 2025"})

[{'title': 'MotorBeam on Instagram: "India’s Top 25 Selling Cars – November 2025 🚗\n\nCompact SUV domination continues as the Tata Nexon leads the charts again with 22,434 units, marking its third straight month above 22k!\n\nMaruti’s Dzire and Swift follow with massive YoY growth, while Tata Punch and Hyundai Creta complete a strong Top 5.\n\nWith 14 SUVs in the Top 25, India’s love for SUVs shows no signs of slowing down. Mahindra’s SUV lineup remains solid and Maruti continues its wide-segment dominance despite pressure on Baleno and Brezza.\n\nWhich car are you rooting for?"',
  'url': 'https://www.instagram.com/p/DSH0_0aAT77/',
  'content': 'Image 27: Here are some detailed shots of the BMW F450GS!\n\nImage 28: BMW F450GS Launched @ ₹4.70 Lakhs! - 452cc Twin Cylinder - \u206048 BHP, 43 Nm - 4 Riding Modes - \u20603 Variants - Base, Exclusive and Trophy - \u20606.5 Inch TFT Display - \u2060ERC - Easy Ride Clutch only for Trophy - \u2060Base - ₹4.70 Lakhs - Exclusive - ₹4.90 Lakhs -

In [ ]:
from langchain import hub

from langchain.agents import AgentExecutor,create_openai_functions_agent

In [ ]:
instructions = """You are an assistant."""
base_prompt = hub.pull("langchain-ai/openai-functions-template")
prompt = base_prompt.partial(instructions=instructions)

In [ ]:
tavily_tool = TavilySearchResults()
tools = [tavily_tool]

agent = create_openai_functions_agent(llm, tools, prompt)

In [ ]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tool,
    verbose=True
)

In [ ]:
print(agent_executor.invoke({"input": "Highest selling suv in India in 2025"}))

### Showcase that one more agent also
### Create our custom agent and custom tools

In [102]:
from pydantic import BaseModel, Field
from langchain_core.tools import BaseTool, StructuredTool, tool
from pydantic import BaseModel

In [103]:
@tool
def search(query:str) -> str:
    """Look up things online."""
    return "LangChain"

In [104]:
print(search.name)
print(search.description)
print(search.args)

search
Look up things online.
{'query': {'title': 'Query', 'type': 'string'}}


In [108]:
@tool
def multiply(a:int, b:int) -> int:
    """Multiply two numbers."""
    return a * b

In [109]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [110]:
class SearchInput(BaseModel):
    query:str = Field(description="should be a search query")

In [111]:
@tool("search-tool", args_schema=SearchInput, return_direct=True)
def search(query:str) -> str:
    """Look up things online."""
    return "LangChain"

In [112]:
print(search.name)
print(search.description)
print(search.args)
print(search.return_direct)

search-tool
Look up things online.
{'query': {'description': 'should be a search query', 'title': 'Query', 'type': 'string'}}
True


In [114]:
class SearchInput(BaseModel):
    query: str = Field(description="should be a search query")

In [115]:
class CalculatorInput(BaseModel):
    a: int = Field(description="first number")
    b: int = Field(description="second number")

### here is my custom tool

In [117]:
from langchain_core.tools import tool
@tool
def get_word_length(word:str) -> int:
    """Returns the length of a word."""
    return len(word)

In [118]:
get_word_length.invoke("abc")

3

In [119]:
tools = [get_word_length]